# Transformer Task

[video](https://www.youtube.com/watch?v=U0s0f995w14#:~:text=In%20this%20video%20we%20read%20the%20original,http://www.peterbloem.nl/blog/transformers%20%E2%9D%A4%EF%B8%8F%20Support%20the%20channel%20%E2%9D%A4%EF%B8%8F%20https://www.youtube.com/)

## Code

In [1]:
import torch
import torch.nn as nn

import models.deep_learning.architectures as mynn


# Encoding

In [ ]:
class WordEncodingLayer(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_size: int,
        max_length: int,
        device: torch.device | None = None,
    ):
        super().__init__()
        self.embed_size = embed_size
        self.device = device

        self.word_embedding = nn.Embedding(vocab_size, embed_size)
        self.position_embedding = nn.Embedding(max_length, embed_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        N, seq_length = x.shape
        positions = torch.arange(0, seq_length).expand(N, seq_length).to(self.device)
        return self.word_embedding(x) + self.position_embedding(positions)


In [ ]:
torch.zeros((2, 3), dtype=torch.bool)

tensor([[False, False, False],
        [False, False, False]])

In [7]:
seq_length = 10
N = 5
positions = torch.arange(0, seq_length).expand(N, seq_length)
positions

tensor([[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]])

In [3]:
L = 5
S = 3
attn_bias = torch.zeros(L, S)
temp_mask = torch.ones(L, S, dtype=torch.bool).tril(diagonal=0)
attn_bias.masked_fill_(temp_mask.logical_not(), float("-inf"))


tensor([[0., -inf, -inf],
        [0., 0., -inf],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])

In [4]:
temp_mask

tensor([[ True, False, False],
        [ True,  True, False],
        [ True,  True,  True],
        [ True,  True,  True],
        [ True,  True,  True]])

## Testing

In [ ]:
src_vocab_size = 1000
embed_size = 512
num_layers = 6
heads = 8
forward_expansion = 4
max_length = 100

In [ ]:
# input parameters
N = 3
batch_size = 2
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
dtype = torch.float32

# Encoder Layer parameters
dropout = 0.2
activation = nn.GELU
layer_norm_eps = 1e-5
batch_first = True
norm_first = True
bias = True

## Transformer encoder parameters
num_layers = 3
#  It helps only when norm_first is True,
norm = None  # nn.LayerNorm(d_model).to(device=device,dtype=dtype)

In [ ]:
tf_encl = mynn.TransformerEncoderLayer(
    d_model=embed_size,
    nhead=heads,  # Assuming each head has 64 dimensions
    dim_feedforward=embed_size * forward_expansion,
    dropout=dropout,
    activation_cls=activation,
    layer_norm_eps=layer_norm_eps,
    batch_first=batch_first,
    norm_first=norm_first,
    bias=bias,
    device=device,
    dtype=dtype,
)
tf_enc = mynn.TransformerEncoder(tf_encl, num_layers, norm=norm)


### Evaluation

In [6]:
tf_enc.eval()
tf_enc(x)

tensor([[[-0.0296, -0.0916,  1.0885, -1.0260],
         [ 0.6635,  0.4575,  1.4182, -0.2418],
         [ 0.4502,  0.3450,  0.7149,  0.3737]],

        [[ 0.1837, -0.0224,  1.0295,  1.6342],
         [ 0.3983, -0.1241,  0.9514,  0.5204],
         [ 1.1837,  1.0843,  1.0421,  0.6918]]], device='mps:0',
       grad_fn=<AddBackward0>)

In [7]:
nn_tf_enc.eval()
nn_tf_enc(x)

tensor([[[-0.0296, -0.0916,  1.0885, -1.0260],
         [ 0.6635,  0.4575,  1.4182, -0.2418],
         [ 0.4502,  0.3450,  0.7149,  0.3737]],

        [[ 0.1837, -0.0224,  1.0295,  1.6342],
         [ 0.3983, -0.1241,  0.9514,  0.5204],
         [ 1.1837,  1.0843,  1.0421,  0.6918]]], device='mps:0',
       grad_fn=<AddBackward0>)

### Training

In [11]:
mse = torch.nn.MSELoss()

In [12]:
torch.manual_seed(train_seed)
nn_tf_enc.train()
print(nn_tf_enc(x))
loss = mse(nn_tf_enc(x), x)
print(loss.item())
loss.backward()
nn_tf_enc(x)


tensor([[[ 0.6852, -0.1630,  2.2074, -0.9302],
         [ 0.9349,  0.6778,  1.9027, -0.1390],
         [ 0.8390,  0.4592,  1.5378,  0.0823]],

        [[-0.1834,  0.3312,  0.7525,  2.1237],
         [ 0.7906,  0.1192,  1.3922,  0.4578],
         [ 0.8707,  0.1922,  0.5967,  0.7011]]], device='mps:0',
       grad_fn=<AddBackward0>)
0.6275849342346191


tensor([[[ 0.2741, -0.4894,  1.8790, -1.0957],
         [ 0.7022,  0.7676,  0.8320, -0.2268],
         [ 0.7146,  0.2815,  0.8392,  1.0298]],

        [[ 0.5085,  0.0161,  1.4960,  2.1186],
         [ 0.4879,  0.0038,  1.0066,  1.0014],
         [ 1.1819,  0.7470,  0.2558,  0.4909]]], device='mps:0',
       grad_fn=<AddBackward0>)

In [13]:
torch.manual_seed(train_seed)
tf_enc.train()
print(tf_enc(x))
loss = mse(tf_enc(x), x)
print(loss.item())
loss.backward()
tf_enc(x)

tensor([[[ 0.6852, -0.1630,  2.2074, -0.9302],
         [ 0.9349,  0.6778,  1.9027, -0.1390],
         [ 0.8390,  0.4592,  1.5378,  0.0823]],

        [[-0.1834,  0.3312,  0.7525,  2.1237],
         [ 0.7906,  0.1192,  1.3922,  0.4578],
         [ 0.8707,  0.1922,  0.5967,  0.7011]]], device='mps:0',
       grad_fn=<AddBackward0>)
0.6275849342346191


tensor([[[ 0.2741, -0.4894,  1.8790, -1.0957],
         [ 0.7022,  0.7676,  0.8320, -0.2268],
         [ 0.7146,  0.2815,  0.8392,  1.0298]],

        [[ 0.5085,  0.0161,  1.4960,  2.1186],
         [ 0.4879,  0.0038,  1.0066,  1.0014],
         [ 1.1819,  0.7470,  0.2558,  0.4909]]], device='mps:0',
       grad_fn=<AddBackward0>)